<a href="https://colab.research.google.com/github/hayahanyyy/Bachelor-Thesis/blob/main/Dataset2FIX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [4]:
df= pd.read_csv('AI_Personalized_Learning.csv')

In [5]:
df.head()

,student_id,age,gender,education_level,learning_style,previous_gpa,completed_modules,avg_time_per_module,engagement_score,distraction_events,quiz_accuracy,feedback_score,contextual_difficulty_level,recommended_path,actual_path_followed,path_efficiency_score,final_assessment_score,learning_outcome
0,STU001,20,Male,UG,Visual,3.75,5,42.70,67,4,100,4,Medium,M1→M2→M4,M1→M3→M5,84,69,Excellent
1,STU002,24,Female,UG,Kinesthetic,2.55,3,31.29,65,2,79,2,Medium,M1→M3→M5,M1→M3→M5,83,68,Fail
2,STU003,22,Male,PG,Visual,3.51,6,52.92,84,5,70,1,Medium,M1→M3→M5,M1→M2→M4,77,84,Good
3,STU004,23,Female,High School,Auditory,3.52,4,40.10,79,0,65,3,Medium,M1→M3→M5,M1→M2→M4,78,91,Fair
4,STU005,21,Female,High School,Kinesthetic,3.19,10,35.34,72,3,50,5,Hard,M1→M2→M4,M1→M3→M5,75,68,Fair


Dropping useless columns

In [6]:
df = df.drop(columns=['student_id'], errors='ignore')

Checking missing values

In [7]:
print("\nMissing values:\n", df.isnull().sum())


Missing values:
 age                            0
gender                         0
education_level                0
learning_style                 0
previous_gpa                   0
completed_modules              0
avg_time_per_module            0
engagement_score               0
distraction_events             0
quiz_accuracy                  0
feedback_score                 0
contextual_difficulty_level    0
recommended_path               0
actual_path_followed           0
path_efficiency_score          0
final_assessment_score         0
learning_outcome               0
dtype: int64


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          1000 non-null   int64  
 1   gender                       1000 non-null   object 
 2   education_level              1000 non-null   object 
 3   learning_style               1000 non-null   object 
 4   previous_gpa                 1000 non-null   float64
 5   completed_modules            1000 non-null   int64  
 6   avg_time_per_module          1000 non-null   float64
 7   engagement_score             1000 non-null   int64  
 8   distraction_events           1000 non-null   int64  
 9   quiz_accuracy                1000 non-null   int64  
 10  feedback_score               1000 non-null   int64  
 11  contextual_difficulty_level  1000 non-null   object 
 12  recommended_path             1000 non-null   object 
 13  actual_path_followe

Feature Engineering

In [18]:
# FEATURE ENGINEERING
# =========================

# Combine performance-related features
df['performance_score'] = (
    df['quiz_accuracy'] + # Using quiz_accuracy as a proxy for quiz_avg_score
    df['final_assessment_score'] + # Using final_assessment_score as a proxy for assignment_completion_rate
    df['engagement_score']
) / 3

# Study efficiency
df['study_efficiency'] = df['quiz_accuracy'] / (df['avg_time_per_module'] + 1) # Using quiz_accuracy and avg_time_per_module

# Engagement intensity
df['engagement_intensity'] = (
    df['engagement_score'] * df['path_efficiency_score'] # Using path_efficiency_score as a proxy for assignment_completion_rate
)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          1000 non-null   int64  
 1   gender                       1000 non-null   object 
 2   education_level              1000 non-null   object 
 3   learning_style               1000 non-null   object 
 4   previous_gpa                 1000 non-null   float64
 5   completed_modules            1000 non-null   int64  
 6   avg_time_per_module          1000 non-null   float64
 7   engagement_score             1000 non-null   int64  
 8   distraction_events           1000 non-null   int64  
 9   quiz_accuracy                1000 non-null   int64  
 10  feedback_score               1000 non-null   int64  
 11  contextual_difficulty_level  1000 non-null   object 
 12  recommended_path             1000 non-null   object 
 13  actual_path_followe

In [20]:
# 4. DEFINE FEATURES & TARGET
# =========================
X = df.drop(columns=['learning_outcome'])
y = df['learning_outcome']

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# Data Preprocessing

Separate column types

In [21]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

print("\nCategorical:", cat_cols)
print("Numerical:", num_cols)


Categorical: ['gender', 'education_level', 'learning_style', 'contextual_difficulty_level', 'recommended_path', 'actual_path_followed']
Numerical: ['age', 'previous_gpa', 'completed_modules', 'avg_time_per_module', 'engagement_score', 'distraction_events', 'quiz_accuracy', 'feedback_score', 'path_efficiency_score', 'final_assessment_score', 'performance_score', 'study_efficiency', 'engagement_intensity']


In [39]:
# Separate column types using the current X
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

# Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())   # IMPORTANT for SVM & Logistic
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

Training and testing split (80-20)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Modelling

Initiating models

In [24]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "SVM": SVC(kernel='rbf')
}

Train and evaluate

In [25]:
for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.21
              precision    recall  f1-score   support

   Excellent       0.26      0.26      0.26        53
        Fail       0.21      0.20      0.21        50
        Fair       0.23      0.19      0.21        47
        Good       0.15      0.18      0.16        50

    accuracy                           0.21       200
   macro avg       0.21      0.21      0.21       200
weighted avg       0.21      0.21      0.21       200


===== Random Forest =====
Accuracy: 0.245
              precision    recall  f1-score   support

   Excellent       0.25      0.30      0.27        53
        Fail       0.25      0.26      0.26        50
        Fair       0.26      0.21      0.23        47
        Good       0.22      0.20      0.21        50

    accuracy                           0.24       200
   macro avg       0.24      0.24      0.24       200
weighted avg       0.24      0.24      0.24       200


===== SVM =====
Accuracy: 0.26
       

Added new evaluation model

In [26]:
from xgboost import XGBClassifier

In [27]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "SVM": SVC(kernel='rbf'),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='mlogloss'
    )
}

In [28]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(random_state=42, eval_metric='mlogloss'))
])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 6, 'model__n_estimators': 300, 'model__subsample': 1.0}
Accuracy: 0.27
              precision    recall  f1-score   support

           0       0.27      0.30      0.29        53
           1       0.30      0.32      0.31        50
           2       0.25      0.23      0.24        47
           3       0.26      0.22      0.24        50

    accuracy                           0.27       200
   macro avg       0.27      0.27      0.27       200
weighted avg       0.27      0.27      0.27       200



Used Ordinal Encoding

In [29]:
from sklearn.preprocessing import OrdinalEncoder

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

In [30]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(random_state=42, eval_metric='mlogloss'))
])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 6, 'model__n_estimators': 300, 'model__subsample': 1.0}
Accuracy: 0.27
              precision    recall  f1-score   support

           0       0.27      0.30      0.29        53
           1       0.30      0.32      0.31        50
           2       0.25      0.23      0.24        47
           3       0.26      0.22      0.24        50

    accuracy                           0.27       200
   macro avg       0.27      0.27      0.27       200
weighted avg       0.27      0.27      0.27       200



In [31]:
for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.21
              precision    recall  f1-score   support

   Excellent       0.26      0.26      0.26        53
        Fail       0.21      0.20      0.21        50
        Fair       0.23      0.19      0.21        47
        Good       0.15      0.18      0.16        50

    accuracy                           0.21       200
   macro avg       0.21      0.21      0.21       200
weighted avg       0.21      0.21      0.21       200


===== Random Forest =====
Accuracy: 0.245
              precision    recall  f1-score   support

   Excellent       0.25      0.30      0.27        53
        Fail       0.25      0.26      0.26        50
        Fair       0.26      0.21      0.23        47
        Good       0.22      0.20      0.21        50

    accuracy                           0.24       200
   macro avg       0.24      0.24      0.24       200
weighted avg       0.24      0.24      0.24       200


===== SVM =====
Accuracy: 0.26
       

Forcing Learnable Target

In [42]:
# =========================
# FORCE LEARNABLE TARGET
# =========================

df['performance_score'] = (
    df['quiz_accuracy'] + # Using quiz_accuracy as a proxy for quiz_avg_score
    df['final_assessment_score'] + # Using final_assessment_score as a proxy for assignment_completion_rate
    df['engagement_score']
) / 3

#df['learning_outcome'] = pd.cut(
 #   df['performance_score'],
  #  bins=[0, 50, 65, 80, 100],
   # labels=['Fail', 'Fair', 'Good', 'Excellent']
#)

# =========================
# BALANCED 4 CLASSES
# =========================

df['learning_outcome'] = pd.qcut(
    df['performance_score'],
    q=4,
    labels=['Fail', 'Fair', 'Good', 'Excellent']
)

Drop old target

In [34]:
X = df.drop(columns=['learning_outcome', 'performance_score'])
y = df['learning_outcome']

Re-encoding target

In [43]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [37]:
from sklearn.preprocessing import OrdinalEncoder

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

In [41]:
for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.965
              precision    recall  f1-score   support

   Excellent       0.97      0.96      0.97        81
        Fair       1.00      0.71      0.83         7
        Good       0.96      0.98      0.97       112

    accuracy                           0.96       200
   macro avg       0.98      0.89      0.92       200
weighted avg       0.97      0.96      0.96       200


===== Random Forest =====
Accuracy: 0.865
              precision    recall  f1-score   support

   Excellent       0.89      0.86      0.88        81
        Fair       0.00      0.00      0.00         7
        Good       0.85      0.92      0.88       112

    accuracy                           0.86       200
   macro avg       0.58      0.59      0.59       200
weighted avg       0.84      0.86      0.85       200


===== SVM =====
Accuracy: 0.915
              precision    recall  f1-score   support

   Excellent       0.95      0.93      0.94        81
    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Accuracy: 0.935
              precision    recall  f1-score   support

   Excellent       0.92      0.95      0.93        81
        Fair       1.00      0.71      0.83         7
        Good       0.95      0.94      0.94       112

    accuracy                           0.94       200
   macro avg       0.95      0.87      0.90       200
weighted avg       0.94      0.94      0.93       200



Re-running after fixing classes

In [45]:
for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    # The error indicates that le.classes_ was an array of integers (e.g., [0, 1, 2]).
    # Based on the previous successful run, the actual string labels were 'Excellent', 'Fair', 'Good'.
    # Explicitly providing these string labels to target_names to resolve the TypeError.
    print(classification_report(y_test, y_pred, target_names=['Excellent', 'Fair', 'Good']))


===== Logistic Regression =====
Accuracy: 0.965
              precision    recall  f1-score   support

   Excellent       0.97      0.96      0.97        81
        Fair       1.00      0.71      0.83         7
        Good       0.96      0.98      0.97       112

    accuracy                           0.96       200
   macro avg       0.98      0.89      0.92       200
weighted avg       0.97      0.96      0.96       200


===== Random Forest =====
Accuracy: 0.865
              precision    recall  f1-score   support

   Excellent       0.89      0.86      0.88        81
        Fair       0.00      0.00      0.00         7
        Good       0.85      0.92      0.88       112

    accuracy                           0.86       200
   macro avg       0.58      0.59      0.59       200
weighted avg       0.84      0.86      0.85       200


===== SVM =====
Accuracy: 0.915
              precision    recall  f1-score   support

   Excellent       0.95      0.93      0.94        81
    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Accuracy: 0.935
              precision    recall  f1-score   support

   Excellent       0.92      0.95      0.93        81
        Fair       1.00      0.71      0.83         7
        Good       0.95      0.94      0.94       112

    accuracy                           0.94       200
   macro avg       0.95      0.87      0.90       200
weighted avg       0.94      0.94      0.93       200



In [46]:
print(df['learning_outcome'].value_counts(dropna=False))

learning_outcome
Fail         255
Fair         252
Excellent    250
Good         243
Name: count, dtype: int64


In [47]:
df['learning_outcome'] = pd.qcut(
    df['performance_score'].rank(method='first'),
    q=4,
    labels=['Fail', 'Fair', 'Good', 'Excellent']
)

In [48]:
print(df['learning_outcome'].value_counts())

learning_outcome
Fail         250
Fair         250
Good         250
Excellent    250
Name: count, dtype: int64


Dropping old target

In [49]:
X = df.drop(columns=['learning_outcome', 'performance_score'])
y = df['learning_outcome']

RE-encode target

In [50]:

le = LabelEncoder()
y = le.fit_transform(y)

Re-doing split

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [53]:
for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    # The error indicates that le.classes_ was an array of integers (e.g., [0, 1, 2]).
    # Based on the previous successful run, the actual string labels were 'Excellent', 'Fair', 'Good'.
    # Explicitly providing these string labels to target_names to resolve the TypeError.
    print(classification_report(y_test, y_pred, target_names=['Excellent', 'Fair', 'Fail', 'Good']))


===== Logistic Regression =====
Accuracy: 0.925
              precision    recall  f1-score   support

   Excellent       0.98      0.88      0.93        50
        Fair       0.94      0.96      0.95        50
        Fail       0.94      0.90      0.92        50
        Good       0.86      0.96      0.91        50

    accuracy                           0.93       200
   macro avg       0.93      0.92      0.93       200
weighted avg       0.93      0.93      0.93       200


===== Random Forest =====
Accuracy: 0.79
              precision    recall  f1-score   support

   Excellent       0.87      0.92      0.89        50
        Fair       0.85      0.88      0.86        50
        Fail       0.70      0.66      0.68        50
        Good       0.73      0.70      0.71        50

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.79       200
weighted avg       0.79      0.79      0.79       200


===== SVM =====
Accuracy: 0.845
      